In [0]:
SELECT
  MIN(PDSI) as PDSI_min, MAX(PDSI) as PDSI_max,
  MIN(MaxT) as MaxT_min, MAX(MaxT) as MaxT_max,
  MIN(MinT) as MinT_min, MAX(MinT) as MinT_max,
  MIN(Precipitation) as Precip_min, MAX(Precipitation) as Precip_max,
  MIN(Cooling) as Cooling_min, MAX(Cooling) as Cooling_max,
  MIN(Heating) as Heating_min, MAX(Heating) as Heating_max,
  MIN(AvgT) as AvgT_min, MAX(AvgT) as AvgT_max
FROM workspace.default.master;

PDSI_min,PDSI_max,MaxT_min,MaxT_max,MinT_min,MinT_max,Precip_min,Precip_max,Cooling_min,Cooling_max,Heating_min,Heating_max,AvgT_min,AvgT_max
-5.56,6.98,73.5,86.8,52.7,65.9,9.11,37.91,424,1713,25,729,63.4,76.0


In [0]:
%python
# Step 1: Read min/max values
minmax_df = spark.sql("SELECT MIN(PDSI) as PDSI_min, MAX(PDSI) as PDSI_max, MIN(MaxT) as MaxT_min, MAX(MaxT) as MaxT_max, MIN(MinT) as MinT_min, MAX(MinT) as MinT_max, MIN(Precipitation) as Precip_min, MAX(Precipitation) as Precip_max, MIN(Cooling) as Cooling_min, MAX(Cooling) as Cooling_max, MIN(Heating) as Heating_min, MAX(Heating) as Heating_max, MIN(AvgT) as AvgT_min, MAX(AvgT) as AvgT_max FROM workspace.default.master")
minmax = minmax_df.collect()[0].asDict()

import numpy as np
from itertools import product

PDSI_vals = np.linspace(minmax['PDSI_min'], minmax['PDSI_max'], 10)
MaxT_vals = np.linspace(minmax['MaxT_min'], minmax['MaxT_max'], 10)
MinT_vals = np.linspace(minmax['MinT_min'], minmax['MinT_max'], 10)
Precip_vals = np.linspace(minmax['Precip_min'], minmax['Precip_max'], 10)
Cooling_vals = np.linspace(minmax['Cooling_min'], minmax['Cooling_max'], 10)
Heating_vals = np.linspace(minmax['Heating_min'], minmax['Heating_max'], 10)
AvgT_vals = np.linspace(minmax['AvgT_min'], minmax['AvgT_max'], 10)

rows = []
for vals in product(PDSI_vals, MaxT_vals, MinT_vals, Precip_vals, Cooling_vals, Heating_vals, AvgT_vals):
    rows.append({
        'PDSI': round(vals[0], 4),
        'MaxT': round(vals[1], 4),
        'MinT': round(vals[2], 4),
        'Precipitation': round(vals[3], 4),
        'Cooling': round(vals[4], 4),
        'Heating': round(vals[5], 4),
        'AvgT': round(vals[6], 4)
    })

import pyspark.sql.types as T
sparkDF = spark.createDataFrame(rows, schema=T.StructType([
    T.StructField('PDSI', T.DoubleType()),
    T.StructField('MaxT', T.DoubleType()),
    T.StructField('MinT', T.DoubleType()),
    T.StructField('Precipitation', T.DoubleType()),
    T.StructField('Cooling', T.DoubleType()),
    T.StructField('Heating', T.DoubleType()),
    T.StructField('AvgT', T.DoubleType())
]))

from pyspark.ml.feature import VectorAssembler
features = ['MaxT', 'MinT', 'Precipitation', 'Cooling', 'Heating', 'AvgT']
assembler = VectorAssembler(inputCols=features, outputCol='features')
sparkDF = assembler.transform(sparkDF)

import mlflow
corn_model = mlflow.spark.load_model("runs:/ae3e1476189d43628f8cf94b7c981e1f/rf_Corn_Yield_model", dfs_tmpdir="/Volumes/workspace/default/databrickshackathon")
soybean_model = mlflow.spark.load_model("runs:/59c1ff8978b4419184c5c9f3e6554f52/rf_Soybean_Yield_model", dfs_tmpdir="/Volumes/workspace/default/databrickshackathon")

corn_preds = corn_model.transform(sparkDF)
soybean_preds = soybean_model.transform(sparkDF)

# Combine predictions (assuming outputCol='prediction' as default)
sparkDF = sparkDF.withColumn('Corn_Yield', corn_preds['prediction'])
sparkDF = sparkDF.withColumn('Soybean_Yield', soybean_preds['prediction'])

# Save table
sparkDF.select('PDSI', 'MaxT', 'MinT', 'Precipitation', 'Cooling', 'Heating', 'AvgT', 'Corn_Yield', 'Soybean_Yield') \
    .write.format('delta').mode('overwrite').saveAsTable('workspace.default.predicted_yields_grid')

# Display a sample
display(sparkDF.select('PDSI', 'MaxT', 'MinT', 'Precipitation', 'Cooling', 'Heating', 'AvgT', 'Corn_Yield', 'Soybean_Yield').limit(100))

INFO:py4j.clientserver:Received command c on object id p0


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can